In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
base_path = os.path.join("..", "data", "data_hm")
files_lst = os.listdir(base_path)
print(f"Users in dataset: {files_lst}")

Users in dataset: ['.DS_Store', '007', '000', '009', '008', '001', '006', '010', '003', '004', '005', '002']


In [4]:
len(files_lst)

12

In [5]:
users = sorted(u for u in os.listdir(base_path) if u.isdigit())[:50]

all_data = []
for user in users:
    traj_path = os.path.join(base_path, user, "Trajectory")
    if not os.path.exists(traj_path):
        continue
    for file in os.listdir(traj_path):
        if file.endswith(".plt"):
            d = pd.read_csv(
                os.path.join(traj_path, file),
                skiprows=6, header=None,
                names=["lat", "lon", "unused", "altitude", "days", "date", "time"],
            )
            d["datetime"] = pd.to_datetime(d["date"] + " " + d["time"])
            d["user"] = user
            all_data.append(d)

full_df = pd.concat(all_data, ignore_index=True)
print(full_df.shape)

(2781512, 9)


In [6]:
full_df[full_df["user"] == "001"]

,lat,lon,unused,altitude,days,date,time,datetime,user
173870,40.013794,116.306530,0,139,39796.997141,2008-12-14,23:55:53,2008-12-14 23:55:53,001
173871,40.013833,116.306538,0,144,39796.997199,2008-12-14,23:55:58,2008-12-14 23:55:58,001
173872,40.013967,116.306339,0,101,39797.001053,2008-12-15,00:01:31,2008-12-15 00:01:31,001
173873,40.014051,116.306288,0,102,39797.001100,2008-12-15,00:01:35,2008-12-15 00:01:35,001
173874,40.014113,116.306160,0,113,39797.001157,2008-12-15,00:01:40,2008-12-15 00:01:40,001
...,...,...,...,...,...,...,...,...,...
282472,40.013804,116.306532,0,90,39744.502928,2008-10-23,12:04:13,2008-10-23 12:04:13,001
282473,40.013803,116.306532,0,90,39744.502986,2008-10-23,12:04:18,2008-10-23 12:04:18,001
282474,40.013803,116.306532,0,90,39744.503044,2008-10-23,12:04:23,2008-10-23 12:04:23,001
282475,40.013803,116.306532,0,90,39744.503079,2008-10-23,12:04:26,2008-10-23 12:04:26,001


In [7]:

def haversine_m(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6_371_000 * np.arcsin(np.sqrt(a))

def detect_stay_points(traj, dist_m=200, time_s=20*60):
    lat, lon = traj["lat"].to_numpy(), traj["lon"].to_numpy()
    t = traj["datetime"].to_numpy()
    n, sp, i = len(traj), [], 0
    while i < n - 1:
        j = i + 1
        while j < n and haversine_m(lat[i], lon[i], lat[j], lon[j]) <= dist_m:
            j += 1
        if (t[j-1] - t[i]) / np.timedelta64(1, "s") >= time_s:
            sp.append({"lat": lat[i:j].mean(), "lon": lon[i:j].mean()})
            i = j
        else:
            i += 1
    return pd.DataFrame(sp)